In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    roc_curve, precision_recall_curve
)
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import json
import pickle

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

PyTorch: 2.11.0+cu130
CUDA disponible: True


In [2]:
BASE_DATA_DIR = '../data'
OUTPUT_DIR    = '../data/processed'
MODELS_DIR    = '../models'
RESULTS_DIR   = '../results'

SEQ_LEN              = 12   # Horas de historia por secuencia
PREDICTION_HORIZON_H = 6    # Predice onset en las próximas 6 horas
MIN_HOURS            = SEQ_LEN  # Mínimo de horas con datos para incluir estancia

# Hiperparámetros
BATCH_SIZE    = 256   # Mayor batch para el dataset más grande
EPOCHS        = 100
LR            = 5e-4
WEIGHT_DECAY  = 1e-4
HIDDEN_SIZE   = 64
NUM_LAYERS    = 1
DROPOUT       = 0.3
PATIENCE      = 12

RANDOM_STATE_SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')

Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

Dispositivo: cuda


In [3]:
features_all = pd.read_parquet(f'{OUTPUT_DIR}/features_all.parquet')
cohort_all   = pd.read_parquet(f'{OUTPUT_DIR}/cohort_all.parquet')

FEATURE_COLS = [
    c for c in features_all.columns
    if c not in ['stay_id', 'hour_bucket']
]
N_FEATURES = len(FEATURE_COLS)

# Calcular onset_h: horas desde intime hasta el onset (solo para sepsis)
cohort_all['intime']       = pd.to_datetime(cohort_all['intime'])
cohort_all['sepsis_onset'] = pd.to_datetime(cohort_all['sepsis_onset'])
cohort_all['onset_h'] = np.where(
    cohort_all['label'] == 1,
    (cohort_all['sepsis_onset'] - cohort_all['intime']).dt.total_seconds() / 3600,
    np.nan
)

ONSET_H_MAP = cohort_all.set_index('stay_id')['onset_h'].to_dict()
LABEL_MAP   = cohort_all.set_index('stay_id')['label'].to_dict()

print(f'Estancias totales : {cohort_all["stay_id"].nunique():,}')
print(f'  Sepsis  (label=1): {(cohort_all["label"]==1).sum():,}')
print(f'  Control (label=0): {(cohort_all["label"]==0).sum():,}')
print(f'Features ({N_FEATURES}): {FEATURE_COLS}')

Estancias totales: 74,754
  Sepsis  (label=1): 37,377
  Control (label=0): 37,377
Features (27): ['Arterial Blood Pressure mean', 'GCS - Motor Response', 'GCS - Verbal Response', 'Heart Rate', 'Non Invasive Blood Pressure diastolic', 'Non Invasive Blood Pressure systolic', 'O2 saturation pulseoxymetry', 'PEEP set', 'Respiratory Rate', 'Temperature Celsius', 'Bands', 'Bicarbonate', 'Bilirubin, Total', 'Creatinine', 'Glucose', 'Lactate', 'Platelet Count', 'Urea Nitrogen', 'White Blood Cells', 'pH', 'pO2', 'urine_output_ml', 'vasopressor_active', 'mechanical_ventilation', 'age', 'gender_enc', 'charlson_score']


In [4]:
# El split se hace a nivel de estancia para evitar data leakage
# Después, build_sequences expande cada estancia en múltiples secuencias
stays_coverage = features_all.groupby('stay_id')['hour_bucket'].count()
valid_stay_ids = stays_coverage[stays_coverage >= MIN_HOURS].index

stay_labels_v = cohort_all[cohort_all['stay_id'].isin(valid_stay_ids)][
    ['stay_id', 'label', 'subject_id']
].reset_index(drop=True)

all_stays    = stay_labels_v['stay_id'].values
all_labels   = stay_labels_v['label'].values
all_subjects = stay_labels_v['subject_id'].values

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE_SEED)
tv_idx, test_idx = next(gss1.split(all_stays, all_labels, groups=all_subjects))

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE_SEED)
tr_idx, val_idx = next(gss2.split(
    all_stays[tv_idx], all_labels[tv_idx], groups=all_subjects[tv_idx]
))

train_stays = all_stays[tv_idx][tr_idx]
val_stays   = all_stays[tv_idx][val_idx]
test_stays  = all_stays[test_idx]

s2sub   = cohort_all.set_index('stay_id')['subject_id']
tr_sub  = set(s2sub[s2sub.index.isin(train_stays)])
va_sub  = set(s2sub[s2sub.index.isin(val_stays)])
te_sub  = set(s2sub[s2sub.index.isin(test_stays)])

print(f'Train: {len(train_stays):,} estancias | {len(tr_sub):,} pacientes')
print(f'Val:   {len(val_stays):,} estancias | {len(va_sub):,} pacientes')
print(f'Test:  {len(test_stays):,} estancias | {len(te_sub):,} pacientes')

Train: 10,468 estancias | 8,815 pacientes
Val: 3,504 estancias | 2,939 pacientes
Test: 3,490 estancias | 2,939 pacientes


In [5]:
def build_sequences(features_df, stay_ids, feature_cols,
                    onset_h_map, label_map,
                    seq_len=SEQ_LEN, horizon_h=PREDICTION_HORIZON_H):
    """
    Genera secuencias para predicción rodante (rolling prediction).

    Para cada estancia y cada hora t con datos previos al onset:
      - X[i]    : últimas seq_len horas de features hasta t  (seq_len, F) con NaN
      - y[i]    : 1 si onset ocurre en (t, t+horizon_h], 0 si no
      - sid[i]  : stay_id (para garantizar split sin leakage)
      - lead[i] : horas hasta el onset (NaN para controles)

    hour_bucket debe ser >= 0 (horas desde intime).
    """
    from numpy.lib.stride_tricks import sliding_window_view

    F = len(feature_cols)
    feat_grp = dict(tuple(
        features_df[features_df['stay_id'].isin(stay_ids)].groupby('stay_id')
    ))

    X_list, y_list, sid_list, lead_list = [], [], [], []

    for sid in stay_ids:
        if sid not in feat_grp:
            continue

        df_s    = feat_grp[sid].sort_values('hour_bucket')
        is_sep  = label_map.get(sid, 0)
        onset_h = onset_h_map.get(sid)

        # Excluir horas en o tras el onset para casos de sepsis
        if is_sep and onset_h is not None and not pd.isna(onset_h):
            max_h = int(onset_h) - 1
        else:
            max_h = int(df_s['hour_bucket'].max())

        df_s = df_s[df_s['hour_bucket'].between(0, max_h)]
        if df_s['hour_bucket'].nunique() < seq_len:
            continue

        # Grid denso (hora 0 … max_h) con NaN donde no hay medición
        grid = np.full((max_h + 1, F), np.nan, dtype=np.float32)
        hb_vals  = df_s['hour_bucket'].astype(int).values
        feat_vals = df_s[feature_cols].values.astype(np.float32)
        valid_mask = (hb_vals >= 0) & (hb_vals <= max_h)
        grid[hb_vals[valid_mask]] = feat_vals[valid_mask]

        if len(grid) < seq_len:
            continue

        # Ventana deslizante vectorizada → (n_windows, seq_len, F)
        windows = sliding_window_view(grid, (seq_len, F)).reshape(-1, seq_len, F)
        n_w     = len(windows)
        end_h   = np.arange(seq_len - 1, seq_len - 1 + n_w, dtype=np.float32)

        # Etiquetas rodantes
        if is_sep and onset_h is not None and not pd.isna(onset_h):
            h2o    = float(onset_h) - end_h
            labels = ((h2o > 0) & (h2o <= horizon_h)).astype(np.float32)
            leads  = h2o
        else:
            labels = np.zeros(n_w, dtype=np.float32)
            leads  = np.full(n_w, np.nan, dtype=np.float32)

        X_list.append(windows)
        y_list.append(labels)
        sid_list.extend([sid] * n_w)
        lead_list.append(leads)

    if not X_list:
        empty = np.empty((0, seq_len, F), dtype=np.float32)
        return empty, np.array([]), np.array([]), np.array([])

    return (
        np.concatenate(X_list,  axis=0),
        np.concatenate(y_list,  axis=0),
        np.array(sid_list),
        np.concatenate(lead_list, axis=0),
    )


print('Construyendo secuencias rodantes...')
X_train_raw, y_train, sids_train, leads_train = build_sequences(features_all, train_stays, FEATURE_COLS, ONSET_H_MAP, LABEL_MAP)
X_val_raw,   y_val,   sids_val,   leads_val   = build_sequences(features_all, val_stays,   FEATURE_COLS, ONSET_H_MAP, LABEL_MAP)
X_test_raw,  y_test,  sids_test,  leads_test  = build_sequences(features_all, test_stays,  FEATURE_COLS, ONSET_H_MAP, LABEL_MAP)

print(f'X_train: {X_train_raw.shape} | positivos: {y_train.mean():.2%}')
print(f'X_val:   {X_val_raw.shape}   | positivos: {y_val.mean():.2%}')
print(f'X_test:  {X_test_raw.shape}  | positivos: {y_test.mean():.2%}')

X_train: (10468, 48, 27) | positivos: 91.19%
X_val: (3504, 48, 27) | positivos: 91.50%
X_test: (3490, 48, 27) | positivos: 90.49%


In [6]:
def apply_forward_fill_vectorized(X):
    """
    Forward fill vectorizado sobre el eje temporal (axis=1).
    Evita el bucle Python por secuencia usando acumulación de índices.
    """
    n, T, F = X.shape
    X_ff = X.copy()
    for j in range(F):
        arr  = X_ff[:, :, j]                          # (n, T)
        mask = ~np.isnan(arr)                          # True donde hay valor real
        idx  = np.where(mask, np.arange(T), 0)        # índice del último válido, o 0
        np.maximum.accumulate(idx, axis=1, out=idx)   # propagar último índice válido
        X_ff[:, :, j] = arr[np.arange(n)[:, None], idx]
    return X_ff


# Calcular máscaras ANTES del forward fill (refleja mediciones reales)
M_train = (~np.isnan(X_train_raw)).astype(np.float32)
M_val   = (~np.isnan(X_val_raw)).astype(np.float32)
M_test  = (~np.isnan(X_test_raw)).astype(np.float32)

X_train_ff = apply_forward_fill_vectorized(X_train_raw)
X_val_ff   = apply_forward_fill_vectorized(X_val_raw)
X_test_ff  = apply_forward_fill_vectorized(X_test_raw)

pct_nan_antes   = np.isnan(X_train_raw).mean()
pct_nan_despues = np.isnan(X_train_ff).mean()
print(f'NaN antes del ffill:   {pct_nan_antes:.1%}')
print(f'NaN despues del ffill: {pct_nan_despues:.1%}')

NaN antes del ffill:   86.2%
NaN después del ffill: 64.4%


In [7]:
def apply_missingness_mask(X, train_medians=None, external_mask=None):
    """
    Estrategia de imputación con missingness mask:
    1. Construye máscara binaria M: 1 = valor real, 0 = imputado.
    2. Rellena NaN con la mediana global del training (valor neutro).
    3. Concatena [X_imputed | M] → shape (n, T, 2*F).
    La máscara preserva la señal de ausencia como feature explícito.

    external_mask: máscara pre-computada desde el tensor original con NaN.
    Pasar siempre que X haya sido modificado antes (e.g. forward fill),
    para que la máscara refleje mediciones reales y no valores propagados.
    """

    _, _, F = X.shape

    # Usar máscara externa si se proporciona; si no, calcularla desde X
    M = external_mask if external_mask is not None else (~np.isnan(X)).astype(np.float32)

    X_imp = X.copy()

    if train_medians is None:
        flat = X_imp.reshape(-1, F)
        train_medians = np.nanmedian(flat, axis=0)
        train_medians = np.where(np.isnan(train_medians), 0.0, train_medians)

    for j in range(F):
        mask_nan = np.isnan(X_imp[:, :, j])
        X_imp[:, :, j][mask_nan] = train_medians[j]

    X_out = np.concatenate([X_imp, M], axis=2).astype(np.float32)
    return X_out, train_medians

# Pasar la máscara original (calculada antes del ffill) para no corromper la señal de ausencia
X_train_masked, train_medians = apply_missingness_mask(X_train_ff, external_mask=M_train)
X_val_masked,  _ = apply_missingness_mask(X_val_ff,  train_medians, external_mask=M_val)
X_test_masked, _ = apply_missingness_mask(X_test_ff, train_medians, external_mask=M_test)

F = len(FEATURE_COLS)
print(f'Shape tras aplicar mask: {X_train_masked.shape} → input_size al LSTM: {X_train_masked.shape[2]}')

Shape tras aplicar mask: (10468, 48, 54) → input_size al LSTM: 54


In [8]:
# Solo se normalizan los 16 valores clínicos — las 16 máscaras ya son binarias {0,1} y no deben normalizarse
n_tr, T, F2 = X_train_masked.shape
F = F2 // 2  # Los primeros 16 canales son valores, los últimos 16 son máscaras

scaler = StandardScaler()  # Normalización z-score: media 0, desviación estándar 1

def scale_masked(X, scaler, F, fit=False):
    n, T, _ = X.shape
    X_vals = X[:, :, :F].reshape(-1, F)  # Extraer las 16 variables
    X_mask = X[:, :, F:] # Separar las 16 máscaras

    if fit:
        # En train: calcular media y desviación estándar y aplicar la normalización
        X_vals_scaled = scaler.fit_transform(X_vals)
    else:
        # En val y test: aplicar la normalización con los parámetros calculados en train
        # (evita data leakage: val/test no influyen en el cálculo de la normalización)
        X_vals_scaled = scaler.transform(X_vals)

    X_vals_scaled = X_vals_scaled.reshape(n, T, F).astype(np.float32)  # Restaurar forma 3D

    # Volver a concatenar valores normalizados con las máscaras sin tocar
    return np.concatenate([X_vals_scaled, X_mask], axis=2).astype(np.float32)

# Aplicar normalización: fit solo en train, transform en val y test
X_train_norm = scale_masked(X_train_masked, scaler, F, fit=True)
X_val_norm   = scale_masked(X_val_masked,   scaler, F, fit=False)
X_test_norm  = scale_masked(X_test_masked,  scaler, F, fit=False)

N_INPUT = X_train_norm.shape[2]  # 32 = 16 valores normalizados + 16 máscaras binarias
print(f'Input con mascaras binarias al LSTM: {N_INPUT}  (shape: {X_train_norm.shape})')
print(f'Máscaras — rango: [{X_train_norm[:,:,F:].min():.0f}, {X_train_norm[:,:,F:].max():.0f}]')

Input con mascaras binarias al LSTM: 54  (shape: (10468, 48, 54))
Máscaras — rango: [0, 1]


In [9]:
class SepsisDataset(Dataset):
    """Clase que alimenta al modelo durante el entrenamiento"""
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = SepsisDataset(X_train_norm, y_train)
val_ds   = SepsisDataset(X_val_norm,   y_val)
test_ds  = SepsisDataset(X_test_norm,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# pos_weight: con predicción rodante los positivos son minoritarios (~5-15%)
n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos
pos_weight_val = n_neg / max(n_pos, 1)

N_INPUT = X_train_norm.shape[2]
print(f'Input shape: {X_train_norm.shape} → N_INPUT: {N_INPUT}')
print(f'Train batches: {len(train_loader)} x {BATCH_SIZE}')
print(f'Positivos: {n_pos:,} ({n_pos/len(y_train):.1%}) | Negativos: {n_neg:,} | pos_weight: {pos_weight_val:.2f}')

Input shape: (10468, 48, 54) → N_INPUT: 54
Train batches: 163 × 64


In [10]:
class LSTMImproved(nn.Module):
    """BiLSTM con mecanismo de atención sobre el eje temporal.

    Reemplaza el avg+max pooling fijo con atención aprendida:
    el modelo aprende a ponderar cada timestep según su relevancia para la predicción.
    La atención es especialmente útil en series clínicas donde las horas más cercanas
    al momento de predicción suelen ser más informativas.
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int = HIDDEN_SIZE,
        num_layers: int = NUM_LAYERS,
        dropout: float = DROPOUT
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )

        lstm_out_size = hidden_size * 2  # bidireccional: concat forward + backward

        # Atención sobre el eje temporal: una puntuación escalar por timestep
        self.attn = nn.Linear(lstm_out_size, 1)

        self.bn1     = nn.BatchNorm1d(lstm_out_size)
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(lstm_out_size, 64)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.lstm(x)                                # (batch, T, hidden*2)

        # Atención: softmax sobre la dimensión temporal → suma ponderada
        attn_weights = torch.softmax(self.attn(out), dim=1)  # (batch, T, 1)
        context = (attn_weights * out).sum(dim=1)             # (batch, hidden*2)

        context = self.bn1(context)
        context = self.dropout(context)
        context = self.relu(self.fc1(context))
        return self.fc2(context).squeeze(-1)


model = LSTMImproved(input_size=N_INPUT).to(DEVICE)
print(model)

LSTMImproved(
  (lstm): LSTM(54, 64, batch_first=True, bidirectional=True)
  (attn): Linear(in_features=128, out_features=1, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)


In [11]:
def train_model(model, model_name, pos_weight_value):
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # Scheduler usa val_loss (más estable para controlar el LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=3, factor=0.5
    )

    history          = {'train_loss': [], 'val_loss': [], 'val_auroc': []}
    best_val_auroc   = 0.0           # Early stopping sobre AUROC
    patience_counter = 0
    best_path        = f'{MODELS_DIR}/{model_name}.pt'

    for epoch in range(1, EPOCHS + 1):

        # ── Fase entrenamiento ──
        model.train()
        train_loss    = 0.0
        train_samples = 0

        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss    += loss.item() * len(y_b)
            train_samples += len(y_b)

        train_loss /= train_samples

        # ── Fase validación ──
        model.eval()
        val_loss  = 0.0
        val_probs, val_true = [], []

        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
                logits = model(X_b)
                val_loss += criterion(logits, y_b).item() * len(y_b)
                val_probs.extend(torch.sigmoid(logits).cpu().numpy())
                val_true.extend(y_b.cpu().numpy())

        val_loss  /= len(val_loader.dataset)
        val_auroc  = roc_auc_score(val_true, val_probs) if len(set(val_true)) > 1 else 0.0

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auroc'].append(val_auroc)

        # Scheduler usa val_loss
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        if new_lr < old_lr:
            print(f'--- Tasa de aprendizaje reducida a: {new_lr:.2e} ---')

        # Early stopping usa val_auroc — guarda el checkpoint con mejor AUROC
        if val_auroc > best_val_auroc:
            best_val_auroc   = val_auroc
            patience_counter = 0
            torch.save(model.state_dict(), best_path)
        else:
            patience_counter += 1

        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
              f'Val AUROC: {val_auroc:.4f} | LR: {new_lr:.2e} | P: {patience_counter}')

        if patience_counter >= PATIENCE:
            print(f'Early stopping en epoch {epoch}. AUROC no mejoró en {PATIENCE} épocas.')
            break

    print(f'\nMejor val_auroc: {best_val_auroc:.4f}')
    return {
        'history': history,
        'path': best_path,
        'model': model,
        'pos_weight': pos_weight.item()
    }

models = {}

model_name = 'best_lstm_attention'
models[model_name] = train_model(model, model_name, n_neg / max(n_pos, 1))

Epoch   1/100 | Train Loss: 0.0995 | Val Loss: 0.0829 | Val AUROC: 0.8520 | LR: 5.00e-04 | P: 0
Epoch   2/100 | Train Loss: 0.0713 | Val Loss: 0.0658 | Val AUROC: 0.9083 | LR: 5.00e-04 | P: 0
Epoch   3/100 | Train Loss: 0.0598 | Val Loss: 0.0589 | Val AUROC: 0.9259 | LR: 5.00e-04 | P: 0
Epoch   4/100 | Train Loss: 0.0540 | Val Loss: 0.0545 | Val AUROC: 0.9349 | LR: 5.00e-04 | P: 0
Epoch   5/100 | Train Loss: 0.0511 | Val Loss: 0.0541 | Val AUROC: 0.9345 | LR: 5.00e-04 | P: 1
Epoch   6/100 | Train Loss: 0.0487 | Val Loss: 0.0540 | Val AUROC: 0.9379 | LR: 5.00e-04 | P: 0
Epoch   7/100 | Train Loss: 0.0463 | Val Loss: 0.0543 | Val AUROC: 0.9365 | LR: 5.00e-04 | P: 1
Epoch   8/100 | Train Loss: 0.0452 | Val Loss: 0.0530 | Val AUROC: 0.9395 | LR: 5.00e-04 | P: 0
Epoch   9/100 | Train Loss: 0.0427 | Val Loss: 0.0547 | Val AUROC: 0.9401 | LR: 5.00e-04 | P: 0
Epoch  10/100 | Train Loss: 0.0412 | Val Loss: 0.0564 | Val AUROC: 0.9381 | LR: 5.00e-04 | P: 1
Epoch  11/100 | Train Loss: 0.0393 | Val

In [12]:
class GRUModel(nn.Module):
    """GRU bidireccional con atención — arquitectura equivalente a LSTMImproved.

    GRU vs LSTM: combina las puertas de actualización y reset en una sola celda,
    sin estado de celda separado. Menos parámetros (~75% del LSTM equivalente),
    entrena más rápido y generaliza mejor con datasets pequeños.
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int = HIDDEN_SIZE,
        num_layers: int = NUM_LAYERS,
        dropout: float = DROPOUT
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )

        gru_out_size = hidden_size * 2  # bidireccional

        self.attn    = nn.Linear(gru_out_size, 1)
        self.bn1     = nn.BatchNorm1d(gru_out_size)
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(gru_out_size, 64)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.gru(x)                                 # (batch, T, hidden*2)

        attn_weights = torch.softmax(self.attn(out), dim=1)  # (batch, T, 1)
        context = (attn_weights * out).sum(dim=1)             # (batch, hidden*2)

        context = self.bn1(context)
        context = self.dropout(context)
        context = self.relu(self.fc1(context))
        return self.fc2(context).squeeze(-1)


gru_model = GRUModel(input_size=N_INPUT).to(DEVICE)
print(gru_model)

model_name_gru = 'best_gru_attention'
models[model_name_gru] = train_model(gru_model, model_name_gru, n_neg / max(n_pos, 1))

GRUModel(
  (gru): GRU(54, 64, batch_first=True, bidirectional=True)
  (attn): Linear(in_features=128, out_features=1, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)
Epoch   1/100 | Train Loss: 0.0956 | Val Loss: 0.0788 | Val AUROC: 0.8565 | LR: 5.00e-04 | P: 0
Epoch   2/100 | Train Loss: 0.0715 | Val Loss: 0.0649 | Val AUROC: 0.9052 | LR: 5.00e-04 | P: 0
Epoch   3/100 | Train Loss: 0.0621 | Val Loss: 0.0617 | Val AUROC: 0.9171 | LR: 5.00e-04 | P: 0
Epoch   4/100 | Train Loss: 0.0575 | Val Loss: 0.0565 | Val AUROC: 0.9284 | LR: 5.00e-04 | P: 0
Epoch   5/100 | Train Loss: 0.0542 | Val Loss: 0.0556 | Val AUROC: 0.9319 | LR: 5.00e-04 | P: 0
Epoch   6/100 | Train Loss: 0.0512 | Val Loss: 0.0563 | Val AUROC: 0.9324 | LR: 5.00e-04 | P: 0
Epoch   7/100 | Train 

In [13]:
class RNNModel(nn.Module):
    """RNN bidireccional con atención — arquitectura equivalente a GRUModel y LSTMImproved.

    RNN vanilla vs GRU/LSTM: sin puertas de actualización ni estado de celda.
    Menos parámetros y más rápido, pero más susceptible al problema del gradiente
    que desaparece en secuencias largas. Sirve como baseline arquitectónico
    para cuantificar cuánto aportan las puertas recurrentes.
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int = HIDDEN_SIZE,
        num_layers: int = NUM_LAYERS,
        dropout: float = DROPOUT
    ):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
            nonlinearity='tanh',
        )

        rnn_out_size = hidden_size * 2  # bidireccional

        self.attn    = nn.Linear(rnn_out_size, 1)
        self.bn1     = nn.BatchNorm1d(rnn_out_size)
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(rnn_out_size, 64)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.rnn(x)                                 # (batch, T, hidden*2)

        attn_weights = torch.softmax(self.attn(out), dim=1)  # (batch, T, 1)
        context = (attn_weights * out).sum(dim=1)             # (batch, hidden*2)

        context = self.bn1(context)
        context = self.dropout(context)
        context = self.relu(self.fc1(context))
        return self.fc2(context).squeeze(-1)


rnn_model = RNNModel(input_size=N_INPUT).to(DEVICE)
print(rnn_model)

model_name_rnn = 'best_rnn_attention'
models[model_name_rnn] = train_model(rnn_model, model_name_rnn, n_neg / max(n_pos, 1))

RNNModel(
  (rnn): RNN(54, 64, batch_first=True, bidirectional=True)
  (attn): Linear(in_features=128, out_features=1, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)
Epoch   1/100 | Train Loss: 0.0907 | Val Loss: 0.0722 | Val AUROC: 0.8918 | LR: 5.00e-04 | P: 0
Epoch   2/100 | Train Loss: 0.0651 | Val Loss: 0.0633 | Val AUROC: 0.9082 | LR: 5.00e-04 | P: 0
Epoch   3/100 | Train Loss: 0.0581 | Val Loss: 0.0601 | Val AUROC: 0.9207 | LR: 5.00e-04 | P: 0
Epoch   4/100 | Train Loss: 0.0552 | Val Loss: 0.0602 | Val AUROC: 0.9228 | LR: 5.00e-04 | P: 0
Epoch   5/100 | Train Loss: 0.0519 | Val Loss: 0.0542 | Val AUROC: 0.9352 | LR: 5.00e-04 | P: 0
Epoch   6/100 | Train Loss: 0.0502 | Val Loss: 0.0558 | Val AUROC: 0.9317 | LR: 5.00e-04 | P: 1
Epoch   7/100 | Train 

In [ ]:
def show_model_metrics_optimized(model, model_name, model_path, leads=None):
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.eval()

    test_probs, test_true = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            logits = model(X_b.to(DEVICE))
            test_probs.extend(torch.sigmoid(logits).cpu().numpy())
            test_true.extend(y_b.numpy())

    test_probs = np.array(test_probs)
    test_true  = np.array(test_true)

    fpr, tpr, thresholds = roc_curve(test_true, test_probs)
    best_thresh = thresholds[np.argmax(tpr - fpr)]
    preds = (test_probs >= best_thresh).astype(int)

    auroc = roc_auc_score(test_true, test_probs)
    auprc = average_precision_score(test_true, test_probs)

    print(f'=== {model_name} (Umbral Opt: {best_thresh:.3f}) ===')
    print(f'AUROC: {auroc:.4f}  |  AUPRC: {auprc:.4f}')
    print(classification_report(test_true, preds, target_names=['No alerta', 'Alerta']))

    # Tiempo medio de alerta temprana (metrica clave para sistema de alerta)
    if leads is not None:
        leads_arr = np.array(leads)
        # Verdaderos positivos: label=1 (onset en horizonte) y modelo predice 1
        tp_mask = (test_true == 1) & (preds == 1) & ~np.isnan(leads_arr)
        fn_mask = (test_true == 1) & (preds == 0) & ~np.isnan(leads_arr)
        if tp_mask.any():
            print(f'\nTiempo de alerta (TP): {leads_arr[tp_mask].mean():.1f}h antes del onset '
                  f'(mediana: {np.median(leads_arr[tp_mask]):.1f}h)')
        print(f'Alarmas emitidas (TP): {tp_mask.sum():,} | Perdidas (FN): {fn_mask.sum():,}')

    return auroc, auprc

for model_name, values in models.items():
    show_model_metrics_optimized(
        model=values['model'],
        model_name=model_name,
        model_path=values['path'],
        leads=leads_test,
    )

=== LSTM best_lstm_attention (Umbral Opt: 0.738) ===
AUROC: 0.9330
              precision    recall  f1-score   support

     Control       0.37      0.92      0.52       332
      Sepsis       0.99      0.83      0.91      3158

    accuracy                           0.84      3490
   macro avg       0.68      0.87      0.71      3490
weighted avg       0.93      0.84      0.87      3490

=== LSTM best_gru_attention (Umbral Opt: 0.479) ===
AUROC: 0.9387
              precision    recall  f1-score   support

     Control       0.40      0.91      0.55       332
      Sepsis       0.99      0.85      0.92      3158

    accuracy                           0.86      3490
   macro avg       0.69      0.88      0.73      3490
weighted avg       0.93      0.86      0.88      3490

=== LSTM best_rnn_attention (Umbral Opt: 0.512) ===
AUROC: 0.9331
              precision    recall  f1-score   support

     Control       0.37      0.92      0.52       332
      Sepsis       0.99      0.83     